# Smart Meter Electricity Consumption Data Pipeline

## Gold Layer - Aggregations & Analytics

### Objective

The Gold layer transforms the curated Silver data into business-ready datasets for reporting and analytics.

In this notebook we will:

- Read the Silver Delta table
- Create hourly consumption summary
- Create daily consumption summary
- Create monthly consumption summary
- Calculate 7-day Simple Moving Average (SMA)
- Save Gold Delta tables

Technology Used:
- PySpark
- Delta Lake
- Databricks

In [0]:
gold_df=spark.table("workspace.default.silver_meter_readings")

In [0]:
display(gold_df)

meter_id,household_id,timestamp,units_consumed,_ingestion_time,is_valid,city,house_type,avg_daily_consumption
M009,H009,2026-04-01T01:00:00.000Z,0.35,2026-07-15T13:23:43.674Z,true,Pune,Independent,6
M010,H010,2026-04-01T03:00:00.000Z,0.1,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M005,H005,2026-04-01T06:00:00.000Z,4.72,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7
M010,H010,2026-04-01T06:00:00.000Z,1.8,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13
M001,H001,2026-04-01T09:00:00.000Z,3.87,2026-07-15T13:23:43.674Z,true,Jaipur,Independent,8
M004,H004,2026-04-01T09:00:00.000Z,2.8,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T10:00:00.000Z,1.06,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M004,H004,2026-04-01T11:00:00.000Z,1.4,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15
M002,H002,2026-04-01T13:00:00.000Z,0.68,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10
M005,H005,2026-04-01T15:00:00.000Z,0.0,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7


In [0]:
gold_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- household_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- units_consumed: double (nullable = true)
 |-- _ingestion_time: timestamp (nullable = true)
 |-- is_valid: boolean (nullable = true)
 |-- city: string (nullable = true)
 |-- house_type: string (nullable = true)
 |-- avg_daily_consumption: integer (nullable = true)



In [0]:
gold_df.count()

1400

In [0]:
gold_valid_df=gold_df.filter(gold_df.is_valid==True)

In [0]:
gold_valid_df.count()

1380

In [0]:
from pyspark.sql.functions import (hour,to_date,month,year)

In [0]:
gold_valid_df = gold_valid_df \
    .withColumn("reading_date", to_date("timestamp")) \
    .withColumn("reading_hour", hour("timestamp")) \
    .withColumn("reading_month", month("timestamp")) \
    .withColumn("reading_year", year("timestamp"))

In [0]:
display(gold_valid_df)

meter_id,household_id,timestamp,units_consumed,_ingestion_time,is_valid,city,house_type,avg_daily_consumption,reading_date,reading_hour,reading_month,reading_year
M009,H009,2026-04-01T01:00:00.000Z,0.35,2026-07-15T13:23:43.674Z,true,Pune,Independent,6,2026-04-01,1,4,2026
M010,H010,2026-04-01T03:00:00.000Z,0.1,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13,2026-04-01,3,4,2026
M005,H005,2026-04-01T06:00:00.000Z,4.72,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7,2026-04-01,6,4,2026
M010,H010,2026-04-01T06:00:00.000Z,1.8,2026-07-15T13:23:43.674Z,true,Hyderabad,Independent,13,2026-04-01,6,4,2026
M001,H001,2026-04-01T09:00:00.000Z,3.87,2026-07-15T13:23:43.674Z,true,Jaipur,Independent,8,2026-04-01,9,4,2026
M004,H004,2026-04-01T09:00:00.000Z,2.8,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15,2026-04-01,9,4,2026
M002,H002,2026-04-01T10:00:00.000Z,1.06,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10,2026-04-01,10,4,2026
M004,H004,2026-04-01T11:00:00.000Z,1.4,2026-07-15T13:23:43.674Z,true,Jaipur,Apartment,15,2026-04-01,11,4,2026
M002,H002,2026-04-01T13:00:00.000Z,0.68,2026-07-15T13:23:43.674Z,true,Kolkata,Independent,10,2026-04-01,13,4,2026
M005,H005,2026-04-01T15:00:00.000Z,0.0,2026-07-15T13:23:43.674Z,true,Kolkata,Apartment,7,2026-04-01,15,4,2026


## Hourly Consumption Aggregation

This aggregation calculates hourly electricity consumption statistics for each smart meter.

Metrics calculated:
- Average consumption
- Minimum consumption
- Maximum consumption
- Total consumption

In [0]:
from pyspark.sql.functions import avg, min, max, sum

In [0]:
hourly_agg_df = gold_valid_df.groupBy(
    "meter_id",
    "reading_date",
    "reading_hour"
).agg(
    avg("units_consumed").alias("avg_units"),
    min("units_consumed").alias("min_units"),
    max("units_consumed").alias("max_units"),
    sum("units_consumed").alias("total_units")
)

In [0]:
display(hourly_agg_df)

meter_id,reading_date,reading_hour,avg_units,min_units,max_units,total_units
M009,2026-04-01,1,0.35,0.35,0.35,0.35
M010,2026-04-01,3,0.1,0.1,0.1,0.1
M005,2026-04-01,6,4.72,4.72,4.72,4.72
M010,2026-04-01,6,1.8,1.8,1.8,1.8
M001,2026-04-01,9,3.87,3.87,3.87,3.87
M004,2026-04-01,9,2.8,2.8,2.8,2.8
M002,2026-04-01,10,1.06,1.06,1.06,1.06
M004,2026-04-01,11,1.4,1.4,1.4,1.4
M002,2026-04-01,13,0.68,0.68,0.68,0.68
M005,2026-04-01,15,0.0,0.0,0.0,0.0


In [0]:
hourly_agg_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- reading_date: date (nullable = true)
 |-- reading_hour: integer (nullable = true)
 |-- avg_units: double (nullable = true)
 |-- min_units: double (nullable = true)
 |-- max_units: double (nullable = true)
 |-- total_units: double (nullable = true)



In [0]:
hourly_agg_df.count()

1380

## Daily Consumption Aggregation

This aggregation calculates daily electricity consumption for each smart meter.

Metrics:
- Average daily consumption
- Minimum consumption
- Maximum consumption
- Total daily consumption

In [0]:
daily_agg_df = gold_valid_df.groupBy(
    "meter_id",
    "reading_date"
).agg(
    avg("units_consumed").alias("avg_daily_units"),
    min("units_consumed").alias("min_daily_units"),
    max("units_consumed").alias("max_daily_units"),
    sum("units_consumed").alias("total_daily_units")
)

In [0]:
daily_agg_df = gold_valid_df.groupBy(
    "meter_id",
    "reading_date"
).agg(
    avg("units_consumed").alias("avg_daily_units"),
    min("units_consumed").alias("min_daily_units"),
    max("units_consumed").alias("max_daily_units"),
    sum("units_consumed").alias("total_daily_units")
)

In [0]:
display(daily_agg_df)

meter_id,reading_date,avg_daily_units,min_daily_units,max_daily_units,total_daily_units
M009,2026-04-01,2.9295833333333334,0.21,14.17,70.31
M010,2026-04-01,0.9034782608695651,0.0,1.92,20.779999999999998
M005,2026-04-01,3.5925000000000007,0.0,6.93,86.22000000000001
M001,2026-04-01,2.864166666666667,0.41,5.23,68.74000000000001
M004,2026-04-01,1.5220833333333335,0.0,3.03,36.53
M002,2026-04-01,1.1795652173913045,0.1,3.53,27.130000000000003
M003,2026-04-01,1.6130434782608696,0.32,3.29,37.1
M008,2026-04-01,1.0982608695652172,0.13,2.13,25.259999999999994
M008,2026-04-02,1.1291666666666667,0.13,2.11,27.099999999999998
M005,2026-04-02,3.0362499999999994,0.37,6.09,72.86999999999999


In [0]:
daily_agg_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- reading_date: date (nullable = true)
 |-- avg_daily_units: double (nullable = true)
 |-- min_daily_units: double (nullable = true)
 |-- max_daily_units: double (nullable = true)
 |-- total_daily_units: double (nullable = true)



In [0]:
daily_agg_df.count()

60

## Monthly Consumption Aggregation

This aggregation summarizes electricity consumption for each smart meter on a monthly basis.

Metrics:
- Average Monthly Consumption
- Minimum Monthly Consumption
- Maximum Monthly Consumption
- Total Monthly Consumption

In [0]:
monthly_agg_df = gold_valid_df.groupBy(
    "meter_id",
    "reading_year",
    "reading_month"
).agg(
    avg("units_consumed").alias("avg_monthly_units"),
    min("units_consumed").alias("min_monthly_units"),
    max("units_consumed").alias("max_monthly_units"),
    sum("units_consumed").alias("total_monthly_units")
)

In [0]:
display(monthly_agg_df)

meter_id,reading_year,reading_month,avg_monthly_units,min_monthly_units,max_monthly_units,total_monthly_units
M009,2026,4,2.2099270072992687,0.0,14.17,302.7599999999998
M010,2026,4,1.0897080291970807,0.0,8.35,149.29000000000005
M005,2026,4,3.401582733812951,0.0,37.13,472.8200000000002
M001,2026,4,2.8553284671532846,0.0,16.81,391.18
M004,2026,4,1.6267142857142844,0.0,12.81,227.7399999999998
M002,2026,4,1.0473188405797103,0.0,5.0,144.53
M003,2026,4,1.877338129496402,0.0,18.03,260.9499999999999
M008,2026,4,1.1842335766423355,0.0,12.28,162.23999999999998
M006,2026,4,3.1316176470588224,0.0,35.65,425.89999999999986
M007,2026,4,4.217071428571431,0.0,38.37,590.3900000000003


In [0]:
monthly_agg_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- reading_year: integer (nullable = true)
 |-- reading_month: integer (nullable = true)
 |-- avg_monthly_units: double (nullable = true)
 |-- min_monthly_units: double (nullable = true)
 |-- max_monthly_units: double (nullable = true)
 |-- total_monthly_units: double (nullable = true)



In [0]:
monthly_agg_df.count()

10

## 7-Day Simple Moving Average (SMA) Prediction

This section predicts expected electricity consumption using a 7-day Simple Moving Average (SMA).

The moving average smooths daily consumption trends and provides a simple estimate of future electricity usage.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

In [0]:
window_spec = Window.partitionBy(
    "meter_id"
).orderBy(
    "reading_date"
).rowsBetween(-6, 0)

In [0]:
prediction_df = daily_agg_df.withColumn(
    "sma_7day",
    avg("total_daily_units").over(window_spec)
)

In [0]:
display(prediction_df)

meter_id,reading_date,avg_daily_units,min_daily_units,max_daily_units,total_daily_units,sma_7day
M001,2026-04-01,2.864166666666667,0.41,5.23,68.74000000000001,68.74000000000001
M001,2026-04-02,2.893333333333333,0.37,5.49,69.44,69.09
M001,2026-04-03,2.7795454545454543,0.32,5.14,61.15,66.44333333333334
M001,2026-04-04,3.3634782608695653,0.28,16.81,77.36,69.1725
M001,2026-04-05,2.6645833333333333,0.0,4.96,63.95,68.128
M001,2026-04-06,2.5270000000000006,0.0,4.79,50.54000000000001,65.19666666666667
M002,2026-04-01,1.1795652173913045,0.1,3.53,27.130000000000003,27.130000000000003
M002,2026-04-02,1.0304166666666665,0.12,1.97,24.729999999999997,25.93
M002,2026-04-03,1.1591666666666665,0.11,5.0,27.819999999999993,26.56
M002,2026-04-04,0.9741666666666667,0.0,1.98,23.380000000000003,25.765


In [0]:
prediction_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- reading_date: date (nullable = true)
 |-- avg_daily_units: double (nullable = true)
 |-- min_daily_units: double (nullable = true)
 |-- max_daily_units: double (nullable = true)
 |-- total_daily_units: double (nullable = true)
 |-- sma_7day: double (nullable = true)



In [0]:
prediction_df.count()

60

In [0]:
hourly_agg_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_hourly_consumption")

In [0]:
daily_agg_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_daily_consumption")

In [0]:
monthly_agg_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_monthly_consumption")

In [0]:
prediction_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_prediction")

In [0]:
%sql
SHOW TABLES;

database,tableName,isTemporary
default,bronze_household_info,false
default,bronze_meter_readings,false
default,customer-incremental,false
default,customer-master,false
default,customer_incremental,false
default,customer_master_delta,false
default,gold_daily_consumption,false
default,gold_hourly_consumption,false
default,gold_monthly_consumption,false
default,gold_prediction,false


In [0]:
%sql
SELECT * FROM workspace.default.gold_hourly_consumption;

meter_id,reading_date,reading_hour,avg_units,min_units,max_units,total_units
M006,2026-04-02,19,5.06,5.06,5.06,5.06
M002,2026-04-02,23,1.0,1.0,1.0,1.0
M010,2026-04-03,17,0.77,0.77,0.77,0.77
M006,2026-04-05,4,0.73,0.73,0.73,0.73
M001,2026-04-01,3,0.96,0.96,0.96,0.96
M008,2026-04-02,3,0.32,0.32,0.32,0.32
M006,2026-04-03,21,4.34,4.34,4.34,4.34
M008,2026-04-01,5,0.71,0.71,0.71,0.71
M007,2026-04-02,0,0.68,0.68,0.68,0.68
M010,2026-04-04,5,0.69,0.69,0.69,0.69


In [0]:
%sql
SELECT * FROM workspace.default.gold_daily_consumption;

meter_id,reading_date,avg_daily_units,min_daily_units,max_daily_units,total_daily_units
M010,2026-04-01,0.9034782608695651,0.0,1.92,20.779999999999998
M001,2026-04-02,2.893333333333333,0.37,5.49,69.44
M007,2026-04-03,4.760416666666667,0.0,38.37,114.25
M009,2026-04-01,2.9295833333333334,0.21,14.17,70.31
M003,2026-04-01,1.6130434782608696,0.32,3.29,37.1
M008,2026-04-06,1.5552631578947371,0.23,12.28,29.550000000000004
M001,2026-04-05,2.6645833333333333,0.0,4.96,63.95
M001,2026-04-01,2.864166666666667,0.41,5.23,68.74000000000001
M004,2026-04-02,1.5799999999999998,0.0,3.04,37.919999999999995
M009,2026-04-02,2.0095833333333335,0.0,3.7,48.230000000000004


In [0]:
%sql
SELECT * FROM workspace.default.gold_monthly_consumption;

meter_id,reading_year,reading_month,avg_monthly_units,min_monthly_units,max_monthly_units,total_monthly_units
M009,2026,4,2.2099270072992687,0.0,14.17,302.7599999999998
M007,2026,4,4.217071428571431,0.0,38.37,590.3900000000003
M005,2026,4,3.401582733812951,0.0,37.13,472.8200000000002
M003,2026,4,1.877338129496402,0.0,18.03,260.9499999999999
M006,2026,4,3.1316176470588224,0.0,35.65,425.89999999999986
M001,2026,4,2.8553284671532846,0.0,16.81,391.18
M010,2026,4,1.0897080291970807,0.0,8.35,149.29000000000005
M004,2026,4,1.6267142857142844,0.0,12.81,227.7399999999998
M002,2026,4,1.0473188405797103,0.0,5.0,144.53
M008,2026,4,1.1842335766423355,0.0,12.28,162.23999999999998


In [0]:
%sql
SELECT * FROM workspace.default.gold_prediction;

meter_id,reading_date,avg_daily_units,min_daily_units,max_daily_units,total_daily_units,sma_7day
M001,2026-04-01,2.864166666666667,0.41,5.23,68.74000000000001,68.74000000000001
M001,2026-04-02,2.893333333333333,0.37,5.49,69.44,69.09
M001,2026-04-03,2.7795454545454543,0.32,5.14,61.15,66.44333333333334
M001,2026-04-04,3.3634782608695653,0.28,16.81,77.36,69.1725
M001,2026-04-05,2.6645833333333333,0.0,4.96,63.95,68.128
M001,2026-04-06,2.5270000000000006,0.0,4.79,50.54000000000001,65.19666666666667
M002,2026-04-01,1.1795652173913045,0.1,3.53,27.130000000000003,27.130000000000003
M002,2026-04-02,1.0304166666666665,0.12,1.97,24.729999999999997,25.93
M002,2026-04-03,1.1591666666666665,0.11,5.0,27.819999999999993,26.56
M002,2026-04-04,0.9741666666666667,0.0,1.98,23.380000000000003,25.765


In [0]:
hourly_agg_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- reading_date: date (nullable = true)
 |-- reading_hour: integer (nullable = true)
 |-- avg_units: double (nullable = true)
 |-- min_units: double (nullable = true)
 |-- max_units: double (nullable = true)
 |-- total_units: double (nullable = true)



In [0]:
daily_agg_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- reading_date: date (nullable = true)
 |-- avg_daily_units: double (nullable = true)
 |-- min_daily_units: double (nullable = true)
 |-- max_daily_units: double (nullable = true)
 |-- total_daily_units: double (nullable = true)



In [0]:
prediction_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- reading_date: date (nullable = true)
 |-- avg_daily_units: double (nullable = true)
 |-- min_daily_units: double (nullable = true)
 |-- max_daily_units: double (nullable = true)
 |-- total_daily_units: double (nullable = true)
 |-- sma_7day: double (nullable = true)



In [0]:
monthly_agg_df.printSchema()

root
 |-- meter_id: string (nullable = true)
 |-- reading_year: integer (nullable = true)
 |-- reading_month: integer (nullable = true)
 |-- avg_monthly_units: double (nullable = true)
 |-- min_monthly_units: double (nullable = true)
 |-- max_monthly_units: double (nullable = true)
 |-- total_monthly_units: double (nullable = true)



In [0]:
display(spark.table("workspace.default.gold_daily_consumption"))

meter_id,reading_date,avg_daily_units,min_daily_units,max_daily_units,total_daily_units
M010,2026-04-01,0.9034782608695651,0.0,1.92,20.779999999999998
M001,2026-04-02,2.893333333333333,0.37,5.49,69.44
M007,2026-04-03,4.760416666666667,0.0,38.37,114.25
M009,2026-04-01,2.9295833333333334,0.21,14.17,70.31
M003,2026-04-01,1.6130434782608696,0.32,3.29,37.1
M008,2026-04-06,1.5552631578947371,0.23,12.28,29.550000000000004
M001,2026-04-05,2.6645833333333333,0.0,4.96,63.95
M001,2026-04-01,2.864166666666667,0.41,5.23,68.74000000000001
M004,2026-04-02,1.5799999999999998,0.0,3.04,37.919999999999995
M009,2026-04-02,2.0095833333333335,0.0,3.7,48.230000000000004


In [0]:
display(spark.table("workspace.default.gold_prediction"))

meter_id,reading_date,avg_daily_units,min_daily_units,max_daily_units,total_daily_units,sma_7day
M001,2026-04-01,2.864166666666667,0.41,5.23,68.74000000000001,68.74000000000001
M001,2026-04-02,2.893333333333333,0.37,5.49,69.44,69.09
M001,2026-04-03,2.7795454545454543,0.32,5.14,61.15,66.44333333333334
M001,2026-04-04,3.3634782608695653,0.28,16.81,77.36,69.1725
M001,2026-04-05,2.6645833333333333,0.0,4.96,63.95,68.128
M001,2026-04-06,2.5270000000000006,0.0,4.79,50.54000000000001,65.19666666666667
M002,2026-04-01,1.1795652173913045,0.1,3.53,27.130000000000003,27.130000000000003
M002,2026-04-02,1.0304166666666665,0.12,1.97,24.729999999999997,25.93
M002,2026-04-03,1.1591666666666665,0.11,5.0,27.819999999999993,26.56
M002,2026-04-04,0.9741666666666667,0.0,1.98,23.380000000000003,25.765
